In [1]:
!pip install kafka-python beautifulsoup4 requests pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.1/326.1 kB 13.3 MB/s eta 0:00:00


In [5]:
import requests
from bs4 import BeautifulSoup
from kafka import KafkaProducer
import json
import time
import re

print("INICIANDO SCRAPER MASIVO DE REVIEWS (Objetivo: ~5.000 reseñas)...")

# --- CONFIGURACIÓN KAFKA ---
try:
    producer = KafkaProducer(
        bootstrap_servers=['kafka:9092'],
        value_serializer=lambda v: json.dumps(v).encode('utf-8')
    )
    print("   Conectado a Kafka.")
except Exception as e:
    print(f"   Error de conexión Kafka: {e}")
    producer = None

# --- FUNCIÓN PARA EXTRAER RESEÑAS DE LA API ---
def obtener_reviews(appid, titulo):
    # Pedimos 100 reviews (el máximo por petición) para tener volumen real
    url_api = f"https://store.steampowered.com/appreviews/{appid}?json=1&filter=recent&language=all&num_per_page=100"
    
    try:
        res = requests.get(url_api, timeout=10)
        data = res.json()
        
        reviews_enviadas = 0
        if 'reviews' in data:
            for r in data['reviews']:
                review_data = {
                    "appid": appid,
                    "titulo_juego": titulo,
                    "review": r['review'],
                    "voted_up": r['voted_up'], # True=Positiva / False=Negativa
                    "language": r['language'],
                    # Convertimos minutos a horas para que la estadística sea legible
                    "playtime_hours": round(r['author'].get('playtime_forever', 0) / 60, 1) if 'author' in r else 0,
                    "timestamp": time.strftime('%Y-%m-%d %H:%M:%S')
                }
                # Enviamos al topic de reviews
                producer.send('steam_reviews', value=review_data)
                reviews_enviadas += 1
        return reviews_enviadas
    except Exception as e:
        print(f"      Error en API para {titulo}: {e}")
        return 0

# --- FUNCIÓN PRINCIPAL DE SCRAPING ---
def ejecutar_scraping_completo():
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    # Buscamos los "Más vendidos" con la etiqueta "Indie" (tag 492)
    url_busqueda = "https://store.steampowered.com/search/?filter=topsellers&tags=492"
    
    try:
        response = requests.get(url_busqueda, headers=headers)
        soup = BeautifulSoup(response.text, 'html.parser')
        rows = soup.select('.search_result_row')
        
        print(f"2. Se han detectado {len(rows)} juegos en la lista de éxitos.")
        print("3. Extrayendo comentarios de cada juego (esto puede tardar 2-3 minutos)...")

        total_final = 0
        # Recorremos TODOS los juegos encontrados (normalmente 50)
        for i, item in enumerate(rows):
            titulo = item.select_one('.title').text.strip()
            url_juego = item['href']
            
            # Extraer AppID con Regex
            match = re.search(r'/app/(\d+)/', url_juego)
            if match:
                appid = match.group(1)
                num_revs = obtener_reviews(appid, titulo)
                total_final += num_revs
                print(f"   [{i+1}/{len(rows)}] -> {titulo}: {num_revs} reseñas enviadas.")
                
                # Pausa técnica para evitar que Steam nos bloquee por exceso de peticiones
                time.sleep(0.5)
            
        producer.flush()
        print(f"\nPROCESO FINALIZADO.")
        print(f"Total de reseñas inyectadas en el pipeline: {total_final}")

    except Exception as e:
        print(f"Error crítico en el scraping: {e}")

# --- EJECUCIÓN ---
if producer:
    ejecutar_scraping_completo()
else:
    print("No se pudo iniciar el script porque Kafka no está disponible.")

🚀 INICIANDO SCRAPER MASIVO DE REVIEWS (Objetivo: ~5.000 reseñas)...
   ✅ Conectado a Kafka.
2. Se han detectado 50 juegos en la lista de éxitos.
3. Extrayendo comentarios de cada juego (esto puede tardar 2-3 minutos)...
   [1/50] -> Slay the Spire 2: 100 reseñas enviadas.
   [2/50] -> Rust: 100 reseñas enviadas.
   [3/50] -> Kerbal Space Program: 100 reseñas enviadas.
   [4/50] -> Dome Keeper: The Lost Keepers: 53 reseñas enviadas.
   [5/50] -> No Man's Sky: 100 reseñas enviadas.
   [6/50] -> Risk of Rain 2: 100 reseñas enviadas.
   [7/50] -> MOUSE: P.I. For Hire: 0 reseñas enviadas.
   [8/50] -> Retro Rewind - Video Store Simulator: 100 reseñas enviadas.
   [9/50] -> Plague Inc: Evolved: 100 reseñas enviadas.
   [10/50] -> Blue Prince: 100 reseñas enviadas.
   [11/50] -> The Isle: 100 reseñas enviadas.
   [12/50] -> Sol Cesto: 99 reseñas enviadas.
   [13/50] -> My Time at Sandrock: 100 reseñas enviadas.
   [14/50] -> Graveyard Keeper: 100 reseñas enviadas.
   [15/50] -> Phasmophobia: 